# LOAD CONFIG

In [0]:
%run ./00_0_config

In [0]:
%sh pwd

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("sql_location", f"{paths['bronze']}mobile_measurements_batch")

sql_location = dbutils.widgets.get("sql_location")

print(f"SQL lokacija je: {sql_location}")

In [0]:
%sql
select :sql_location as location_param;

# TABLE CREATION mobile_measurements_batch AND INITIAL LOAD

In [0]:
raw_df = spark.read.json(paths['raw'])

raw_df.createOrReplaceTempView("tmp_raw_measurements")

#spark.sql("""DROP TABLE IF EXISTS telco_prod.`01-bronze`.mobile_measurements_batch""")

spark.sql(f"""
CREATE OR REPLACE TABLE telco_prod.`01-bronze`.mobile_measurements_batch
USING DELTA
PARTITIONED BY (processing_date)
LOCATION '{sql_location}'
AS
SELECT *, current_timestamp() as _ingested_at
FROM tmp_raw_measurements
""")

In [0]:
display(spark.sql("DESCRIBE DETAIL telco_prod.`01-bronze`.mobile_measurements_batch"))

In [0]:
display(spark.table("telco_prod.`01-bronze`.mobile_measurements_batch"))

In [0]:
batch_df = spark.table("telco_prod.`01-bronze`.mobile_measurements_batch")
#batch_df.count()
display(batch_df.groupBy("processing_date").count().orderBy("processing_date"))

# INCREMENTAL BATCH LOAD

## MERGE APPROACH

In [0]:
new_data_df = spark.read.json(paths['raw'])
new_data_df.createOrReplaceTempView("v_source")

spark.sql(f"""
MERGE INTO telco_prod.`01-bronze`.mobile_measurements_batch AS target
USING (
  SELECT *, current_timestamp() as _ingested_at FROM v_source
) AS source
ON target.row_id = source.row_id AND target.timestamp = source.timestamp
WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
# check

df_new = spark.table("telco_prod.`01-bronze`.mobile_measurements_batch")
df_new.count()

## JOIN APPROACH

In [0]:
from pyspark.sql.functions import col, current_timestamp

# dates already loaded in bronze table
existing_dates = spark.sql("SELECT DISTINCT processing_date FROM telco_prod.`01-bronze`.mobile_measurements_batch").collect()
list_of_dates = [row['processing_date'] for row in existing_dates]

list_of_dates

# dates in json files but not already loaded
raw_df = spark.read.json(paths['raw']) \
    .filter(~col("processing_date").isin(list_of_dates))
#display(raw_df)

# append only new dates
(raw_df
    .select("*", current_timestamp().alias("_ingested_at")) \
    .write.mode("append") \
    .partitionBy("processing_date") \
    .saveAsTable("telco_prod.`01-bronze`.mobile_measurements_batch")
)